In [15]:
!pip install astsa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 12.1 MB/s eta 0:00:00


In [22]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import astsa
from scipy.interpolate import make_smoothing_spline

# Problem 6.20

Let $y_t$ represent the global temperature series (globtemp) shown in Figure 1.2.

In [20]:
df = astsa.load_gtemp_land()
df.head()

,Time,Value
0,1850,-0.5
1,1851,-0.6
2,1852,-0.5
3,1853,-0.5
4,1854,-0.2


In [21]:
# Recreate Figure 1.2 from R code into python
# plot(globtemp, type="o", ylab="Global Temperature Deviations")

fig = px.line(df, x=df['Time'], y=df['Value'], title='Global Temperature Deviations')
fig = go.Figure(fig)
fig = fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Temperature Deviation',
)
fig.show()

## Part (a)

Fit a smoothing spline using $GCV$ (the default) to $y_t$ and plot the result superimposed on the data. Repeat the fit using `spar=.7`; the $GCV$ method yields `spar=.5` approximately. (Example 2.14 on page 70 may help. Also in R, see the help file `?smooth.spline`.)

Translate the following from R to Python:
``` R
plot(soi)
lines(smooth.spline(time(soi), soi, spar=.5), lwd=2, col=4)
lines(smooth.spline(time(soi), soi, spar=1), lty=2, lwd=2, col=2)
```

In [41]:
# Use Scipy version to make smoothing splines with different spar
spl = make_smoothing_spline(df['Time'], df['Value'])
spl2 = make_smoothing_spline(df['Time'], df['Value'], lam=445.72)

In [42]:
# Plot the original time series with the 2 splines
fig = px.line(df, x=df['Time'], y=df['Value'], title='Global Temperature Deviations')
fig = go.Figure(fig)
fig = fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Temperature Deviation',
)
fig.add_trace(go.Scatter(x=df['Time'], y=spl(df['Time']), mode='lines', name='GCV .5'))
fig.add_trace(go.Scatter(x=df['Time'], y=spl2(df['Time']), mode='lines', name='GCV .7'))
fig.show()

The larger the `spar` in this case, the more interpolated the data and less smooth.

This is opposite to the value of $\lambda$, as the larger the $\lambda$ the smoother it is with smaller values corresponding to higher interpolations.

---

**Correction**

`spar` and $\lambda$ are not the same thing, but instead is monotonically linked via the equation:
$$
\lambda \propto 256^{3s-1}
$$
This makes the 0.5 and 0.7 spar equivalent to the following $\lambda$ below:
$$
256^{3(0.5)-1} = 16\\
256^{3(0.7)-1} = 445.72
$$

## Part (b)

Write the model $y_t = x_t + v_t$ with $\nabla^2 x_t=w_t$, in state-space model to $y_t$, and exhibit a time plot the estimated smoother $\hat{x^n_t}$ and the corresponding error limits, $\hat{x^n_t} \pm 2\sqrt{\hat{p^n_t}}$ superimposed on the data.

The State Space model is represented in the R code following below:
``` R
Phi = matrix(c(2,1,-1,0),2); A = matrix(c(1,0),1)
mu0 = matrix(0,2); Sigma0 = diag(1,2)
Linn = function(para){
  sigw = para[1]; sigv = para[2]
  cQ = diag(c(sigw,0))
  kf = Kfilter0(num, y, A, mu0, Sigma0, Phi, cQ, sigv)
  return(kf$like)
}

init.par = c(.1, 1)
(est = optim(init.par, Linn, NULL, method="BFGS", hessian=TRUE, control=list(trace=1,REPORT=1)))
SE = sqrt(diag(solve(est$hessian)))

estimate = est$par; u = cbind(estimate, SE)
rownames(u) = c("sigw", "sigv"); u

sigw = est$par[1]
cQ = diag(c(sigw,0))
sigv = est$par[2]
ks = Ksmooth0(num, y, A, mu0, Sigma0, Phi, cQ, sigv)
xsmoo = ts(ks$xs[1,1,]); psmoo = ts(ks$Ps[1,1,])
upp = xsmoo+2*sqrt(psmoo); low = xsmoo-2*sqrt(psmoo)
lines(upp, cos
```

In [ ]:
# First define the state-space model from scratch
phi = np.array([[2, 1], [-1, 0]])
a = np.array([1, 0])
mu0 = np.array([0, 0])
sigma0 = np.array([[1, 0], [0, 1]])



## Part (c)

Superimpose all the fits from parts (a) and (b) [include the error bounds] on the data and briefly compare and contrast the results.